In [ ]:
# ============================================================
# 2-PANEL FINAL (Annual only)
# a: detrended (1990–2020)
# b: RAW anomaly full period (1955–2020)
# OBS aligned to ODA over 1990–2020
# ALL ensemble members
# Plot variables stored for reuse
# ============================================================

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.signal import detrend

# ============================================================
# SETTINGS
# ============================================================

SIG_OBS_MIN, SIG_OBS_MAX = 25.2, 25.4
SIG_MOD_MIN, SIG_MOD_MAX = 24.22, 24.73

LAT_MIN, LAT_MAX = 20, 30
LON_MIN, LON_MAX = 136, 138

T0_FULL = "1955-01-01"
T0_OBS  = "1990-01-01"
T1      = "2020-12-31"

MODELS = ["LE","ODA","WDA"]
COLORS = {"LE":"orange","ODA":"blue","WDA":"red"}
OBS_COLOR="black"

# ============================================================
# HELPERS
# ============================================================

def to_month_start(da):
    da = da.copy()
    da["time"] = da.time.astype("datetime64[M]").astype("datetime64[ns]")
    return da

def annual_mean_func(da):
    da = da.groupby("time.year").mean("time")
    da = da.assign_coords(
        time=("year",[np.datetime64(f"{y}-07-01") for y in da.year.values])
    ).swap_dims({"year":"time"}).drop_vars("year")
    return da

def detrend_nan_safe(da):
    y = da.values.astype(float)
    ok = np.isfinite(y)
    y_dt = np.full_like(y,np.nan)
    if ok.sum()>2 and np.nanstd(y[ok])>1e-12:
        y_dt[ok] = detrend(y[ok],type="linear")
    return da.copy(data=y_dt)

def corr_func(obs,mod):
    o,m = xr.align(obs,mod,join="inner")
    ok = np.isfinite(o.values)&np.isfinite(m.values)
    if ok.sum()<10:
        return np.nan
    return np.corrcoef(o.values[ok],m.values[ok])[0,1]

def get_ens_dim(da):
    return next((d for d in da.dims if d.startswith("ens")), None)

# ============================================================
# OBS
# ============================================================

mask_obs = (
    (ds_obs.SIGMA0>=SIG_OBS_MIN)&
    (ds_obs.SIGMA0<=SIG_OBS_MAX)&
    (ds_obs.lat>=LAT_MIN)&
    (ds_obs.lat<=LAT_MAX)&
    (ds_obs.lon>=LON_MIN)&
    (ds_obs.lon<=LON_MAX)
)

dic_obs = (
    ds_obs.DIC.where(mask_obs)
    .mean(("z_t","lat","lon"))
)

dic_obs = to_month_start(dic_obs).sel(time=slice(T0_OBS,T1))

OBS_RAW = annual_mean_func(dic_obs)
OBS_RAW = OBS_RAW - OBS_RAW.mean("time")
OBS_DT  = detrend_nan_safe(OBS_RAW)

# ============================================================
# MODELS
# ============================================================

MOD_RAW={}
MOD_DT={}
SPREAD={}
R={}

for m in MODELS:

    sig = getattr(results["SIGMA0"],f"{m}_ds_rgd")
    dic_ds = getattr(results["DIC"],f"{m}_ds_rgd")
    dic = dic_ds["DIC"] if isinstance(dic_ds,xr.Dataset) else dic_ds

    sig = sig.sel(lat=slice(LAT_MIN,LAT_MAX),
                  lon=slice(LON_MIN,LON_MAX))
    dic = dic.sel(lat=slice(LAT_MIN,LAT_MAX),
                  lon=slice(LON_MIN,LON_MAX))

    ens_dim_sig = get_ens_dim(sig)
    ens_dim_dic = get_ens_dim(dic)

    members = sig[ens_dim_sig].values if ens_dim_sig else [None]

    raw_list=[]
    dt_list=[]
    r_list=[]

    for e in members:

        if e is not None:
            sig_e = sig.sel({ens_dim_sig:e})
            dic_e = dic.sel({ens_dim_dic:e})
        else:
            sig_e = sig
            dic_e = dic

        mask = (sig_e>=SIG_MOD_MIN)&(sig_e<=SIG_MOD_MAX)

        dic_mon = (
            dic_e.where(mask)
            .mean(("z_t","lat","lon"))
        )

        dic_mon = to_month_start(dic_mon)

        if m=="WDA":
            dic_mon = dic_mon.sel(time=slice(T0_FULL,"2014-12-31"))
        else:
            dic_mon = dic_mon.sel(time=slice(T0_FULL,T1))

        ann_raw = annual_mean_func(dic_mon)
        ann_raw = ann_raw - ann_raw.mean("time")
        ann_dt  = detrend_nan_safe(ann_raw)

        raw_list.append(ann_raw)
        dt_list.append(ann_dt)

        r_list.append(
            corr_func(
                OBS_DT,
                ann_dt.sel(time=slice(T0_OBS,T1))
            )
        )

    if ens_dim_sig:
        raw_stack = xr.concat(raw_list,dim="member")
        dt_stack  = xr.concat(dt_list,dim="member")

        MOD_RAW[m] = raw_stack.mean("member")
        MOD_DT[m]  = dt_stack.mean("member")
        SPREAD[m]  = dt_stack.std("member")
        R[m]       = (np.nanmean(r_list),np.nanstd(r_list))
    else:
        MOD_RAW[m] = raw_list[0]
        MOD_DT[m]  = dt_list[0]
        SPREAD[m]  = None
        R[m]       = (r_list[0],0)

# ============================================================
# OBS ALIGN TO ODA
# ============================================================

offset = (
    MOD_RAW["ODA"]
    .sel(time=slice(T0_OBS,T1))
    .mean("time")
    -
    OBS_RAW.sel(time=slice(T0_OBS,T1)).mean("time")
)

OBS_RAW_ALIGN = OBS_RAW + offset

# ============================================================
# STORE PLOT DATA (for later tuning)
# ============================================================

PLOT_DATA_1AB = {
    "OBS_DT": OBS_DT,
    "OBS_RAW_ALIGN": OBS_RAW_ALIGN,
    "MOD_DT": MOD_DT,
    "MOD_RAW": MOD_RAW,
    "SPREAD": SPREAD,
    "R": R
}

# ============================================================
# PLOT
# ============================================================

fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)

# ---- a detrended
ax=axes[0]

ax.scatter(PLOT_DATA_1AB["OBS_DT"].time,
           PLOT_DATA_1AB["OBS_DT"],
           s=35,color=OBS_COLOR,label="OBS")

for m in MODELS:
    mean=PLOT_DATA_1AB["MOD_DT"][m].sel(time=slice(T0_OBS,T1))
    std =PLOT_DATA_1AB["SPREAD"][m]

    ax.plot(mean.time,mean,color=COLORS[m],lw=2,label=m)

    if std is not None:
        std=std.sel(time=slice(T0_OBS,T1))
        ax.fill_between(mean.time,(mean-std),(mean+std),
                        color=COLORS[m],alpha=0.25)

txt="\n".join([f"{m}: r={R[m][0]:.2f}±{R[m][1]:.2f}" for m in MODELS])
ax.text(0.02,0.98,txt,transform=ax.transAxes,ha="left",va="top")

ax.text(0.02,0.85,"a",transform=ax.transAxes,
        va="top",fontweight="bold")

ax.set_xlim(np.datetime64(T0_OBS),np.datetime64(T1))
ax.set_title("Annual (detrended)")
ax.set_ylabel("DIC anomaly")
ax.legend()

# ---- b raw
ax=axes[1]

ax.scatter(PLOT_DATA_1AB["OBS_RAW_ALIGN"].time,
           PLOT_DATA_1AB["OBS_RAW_ALIGN"],
           s=25,color=OBS_COLOR,label="OBS")

for m in MODELS:
    ax.plot(PLOT_DATA_1AB["MOD_RAW"][m].time,
            PLOT_DATA_1AB["MOD_RAW"][m],
            color=COLORS[m],lw=2,label=m)

ax.text(0.02,0.98,"b",transform=ax.transAxes,
        va="top",fontweight="bold")

ax.set_xlim(np.datetime64(T0_FULL),np.datetime64(T1))
ax.set_title("Annual (full period)")

plt.tight_layout()
plt.show()
